# ИНН клиентов ТЭ помесячно, 2026-01 … 2026-09, и сверка с отчётами коллег

Adhoc: уникальные ИНН клиентов торгового эквайринга по месяцам, лист Excel на месяц.

**Клиент ТЭ в месяце** — периметр `final_df` (`final_df_script_3_1.ipynb`, секция 01; в DAG — `acq_01_sa_perimeter.sql`):
договор SA, действующий в месяце, с привязкой терминала типа P в этом же месяце, договор и компания не удалены, ИНН заполнен.
Периметр считается запросом на каждый месяц; остальные части `final_df` строк не отсекают, поэтому полный скрипт не нужен.

**ИНН** нормализуется как в 3_1: только цифры, 9 цифр → 10, 11 цифр → 12, другая длина — некорректный ИНН.
Контрольная сумма ФНС только диагностируется, строки по ней не отбрасываются.

**ЧОД ТЭ** — колонка `chod` готового `final_df` от `final_df_script_3_1.ipynb`, сумма по ИНН за месяц.
Источник: period CSV `final_df_period_2026_01_*_final_df_script_3_1.csv`, недостающие месяцы (сентябрь) —
чекпоинт `checkpoints_final_df_2026_01_*_final_df_script_3_1/final_df_2026_MM.parquet|csv`.
Если у ИНН периметра нет строки в `final_df`, ЧОД остаётся пустым.

**Сверка** — с месячными отчётами коллег `Equaring/Data/??_<Месяц>_2026.xlsx` по нормализованному ИНН:
в обоих, только периметр, только отчёт. Для «только отчёт» причина ищется по Альфе.


In [ ]:
import re
import time
from pathlib import Path

import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 220)

imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'},
)
imp._init_connection()
print('Impala connected')

MONTHS = [f'2026-{m:02d}' for m in range(1, 10)]

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
OUT_DIR = Path('/home/jovyan/documents/Equaring/Adhocs/ИНН ТЭ помесячно')
OUT_DIR.mkdir(parents=True, exist_ok=True)
XLSX_PATH = OUT_DIR / 'te_inn_by_month_2026_01_2026_09.xlsx'
CLIENT_XLSX_PATH = OUT_DIR / 'te_inn_chod_2026_01_2026_09.xlsx'

FINAL_DF_CSV_GLOB = 'final_df_period_2026_01_*_final_df_script_3_1.csv'
CHECKPOINT_DIR_GLOB = 'checkpoints_final_df_2026_01_*_final_df_script_3_1'

# Явный путь к отчёту месяца, если автопоиск ??_*_2026.xlsx берёт не тот файл.
EXCEL_OVERRIDE = {}
# Явное имя колонки ИНН в отчёте, если автопоиск ошибается.
EXCEL_INN_COL = None

IN_CHUNK = 1000

# Запросы периметра и причин лёгкие; меньший MEM_LIMIT проще проходит допуск в default-pool.
QUERY_MEM_LIMIT = '4g'
ADMISSION_RETRIES = 3
ADMISSION_WAIT_SEC = 120


def month_bounds(m):
    start = pd.Timestamp(f'{m}-01')
    end = start + pd.offsets.MonthEnd(0)
    return f'{start:%Y-%m-%d}', f'{end:%Y-%m-%d}'


def normalize_inn(v):
    if v is None or pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0+$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def _inn_digit(s, weights):
    return sum(int(d) * w for d, w in zip(s, weights)) % 11 % 10


def inn_checksum_ok(s):
    if not s:
        return False
    if len(s) == 10:
        return _inn_digit(s[:9], [2, 4, 10, 3, 5, 9, 4, 6, 8]) == int(s[9])
    if len(s) == 12:
        c11 = _inn_digit(s[:10], [7, 2, 4, 10, 3, 5, 9, 4, 6, 8])
        c12 = _inn_digit(s[:11], [3, 7, 2, 4, 10, 3, 5, 9, 4, 6, 8])
        return c11 == int(s[10]) and c12 == int(s[11])
    return False


def _low(v):
    return re.sub(r'\s+', ' ', str(v)).strip().lower()


def pick(cols, exact, contains=None, exclude=()):
    low = {c: _low(c) for c in cols}
    for e in exact:
        for c in cols:
            if low[c] == e:
                return c
    if contains:
        for c in cols:
            if all(k in low[c] for k in contains) and not any(x in low[c] for x in exclude):
                return c
    return None


INN_HEADER_RE = re.compile(r'(^|\W)инн(\W|$)')


def read_with_header(path, sheet=0, max_rows=20):
    raw = pd.read_excel(path, sheet_name=sheet, header=None, dtype=object)
    for i in range(min(max_rows, len(raw))):
        if any(INN_HEADER_RE.search(_low(v)) for v in raw.iloc[i] if pd.notna(v)):
            names, seen = [], {}
            for j, v in enumerate(raw.iloc[i]):
                n = re.sub(r'\s+', ' ', str(v)).strip() if pd.notna(v) else f'col_{j}'
                seen[n] = seen.get(n, 0) + 1
                names.append(n if seen[n] == 1 else f'{n}_{seen[n]}')
            df = raw.iloc[i + 1:].copy()
            df.columns = names
            return df.dropna(how='all').reset_index(drop=True), i
    raise ValueError(f'Не нашёл строку заголовков с «ИНН» в первых {max_rows} строках {path}')


def chunks(items, n):
    items = list(items)
    for i in range(0, len(items), n):
        yield items[i:i + n]


def fetch(sql, label):
    for attempt in range(1, ADMISSION_RETRIES + 1):
        try:
            with imp:
                imp.execute(f'set MEM_LIMIT={QUERY_MEM_LIMIT}')
                return imp.fetch(sql)
        except Exception as exc:
            if 'Admission for query exceeded timeout' not in str(exc) or attempt == ADMISSION_RETRIES:
                raise
            print(f'{label}: не прошёл очередь Impala (попытка {attempt}/{ADMISSION_RETRIES}), '
                  f'жду {ADMISSION_WAIT_SEC} с')
            time.sleep(ADMISSION_WAIT_SEC)


## 1) Периметр на каждый месяц

Один запрос на месяц, условия как в секции 01 `final_df_script_3_1.ipynb`.


In [ ]:
def perimeter_sql(month_start, month_end):
    return f'''
SELECT DISTINCT
  cast(a.n_agr AS string) AS n_agr,
  cast(a.c_agr_number AS string) AS contract_number,
  cast(a.d_valid_from AS date) AS d_valid_from,
  cast(a.d_valid_to AS date) AS d_valid_to,
  cast(c.c_inn AS string) AS inn_raw,
  cast(c.c_cmp_name AS string) AS company_name
FROM ods_alpha.scd1_agreements a
JOIN ods_alpha.scd1_companies c
  ON c.n_cmp = a.n_cmp_client
WHERE upper(trim(cast(a.acq_class AS string))) = 'SA'
  AND cast(a.d_valid_from AS date) <= cast('{month_end}' AS date)
  AND (a.d_valid_to IS NULL OR cast(a.d_valid_to AS date) >= cast('{month_start}' AS date))
  AND coalesce(a.ods_deleted_flg, '0') <> '1'
  AND coalesce(c.ods_deleted_flg, '0') <> '1'
  AND c.c_inn IS NOT NULL
  AND EXISTS (
      SELECT 1
      FROM ods_alpha.scd1_agr_terms t
      WHERE cast(t.n_agr AS string) = cast(a.n_agr AS string)
        AND cast(t.d_valid_from AS date) <= cast('{month_end}' AS date)
        AND (t.d_valid_to IS NULL OR cast(t.d_valid_to AS date) >= cast('{month_start}' AS date))
        AND upper(trim(cast(t.cf_ter_type AS string))) = 'P'
        AND coalesce(t.ods_deleted_flg, '0') <> '1'
  )
'''


# Посчитанные месяцы остаются в памяти: повторный запуск ячейки после ошибки считает только недостающие.
per_by_month = globals().get('per_by_month') or {}
for m in MONTHS:
    if m in per_by_month:
        print(f'{m}: договоров {len(per_by_month[m]):,} (уже посчитан)')
        continue
    month_start, month_end = month_bounds(m)
    part = fetch(perimeter_sql(month_start, month_end), f'периметр {m}')
    part['report_month'] = m
    per_by_month[m] = part
    print(f'{m}: договоров {len(part):,}')

per_df = pd.concat([per_by_month[m] for m in MONTHS], ignore_index=True)
per_df['inn'] = per_df['inn_raw'].map(normalize_inn)
per_df['d_valid_from'] = pd.to_datetime(per_df['d_valid_from'], errors='coerce')
per_df['d_valid_to'] = pd.to_datetime(per_df['d_valid_to'], errors='coerce')
print('строк периметра (договор × месяц):', f'{len(per_df):,}')


## 2) Качество ИНН периметра

ИНН, не ставший 10 или 12 цифрами, в выгрузку не попадает. Неверная контрольная сумма только отмечается.


In [ ]:
per_df['inn_checksum_ok'] = per_df['inn'].map(inn_checksum_ok)

quality_rows = []
bad_norm = per_df.loc[per_df['inn'].isna()]
bad_sum = per_df.loc[per_df['inn'].notna() & ~per_df['inn_checksum_ok']]
for df, problem in ((bad_norm, 'не 10/12 цифр после нормализации'), (bad_sum, 'неверная контрольная сумма')):
    if len(df):
        quality_rows.append(pd.DataFrame({
            'Источник': 'периметр',
            'Месяц': df['report_month'],
            'ИНН сырой': df['inn_raw'],
            'ИНН нормализованный': df['inn'],
            'Проблема': problem,
            'Наименование': df['company_name'],
            'Номер договора': df['contract_number'],
        }))

per_quality = (
    per_df.groupby('report_month')
    .agg(
        contracts=('n_agr', 'nunique'),
        inn_raw=('inn_raw', 'nunique'),
        inn=('inn', 'nunique'),
        bad_norm_rows=('inn', lambda s: int(s.isna().sum())),
        bad_checksum_inn=('inn', lambda s: s[~s.map(inn_checksum_ok) & s.notna()].nunique()),
    )
    .reset_index()
)
display(per_quality)
if len(bad_norm):
    print('Примеры ИНН, не прошедших нормализацию:')
    display(bad_norm[['report_month', 'inn_raw', 'company_name', 'contract_number']].head(20))
if len(bad_sum):
    print('Примеры ИНН с неверной контрольной суммой:')
    display(bad_sum[['report_month', 'inn_raw', 'inn', 'company_name']].drop_duplicates('inn').head(20))

per_inn = per_df.dropna(subset=['inn'])
per_sets = {m: set(per_inn.loc[per_inn['report_month'] == m, 'inn']) for m in MONTHS}


## 2б) ЧОД ТЭ из `final_df` (3_1)

Period CSV даёт январь–август; месяцы, которых в нём нет, берутся из чекпоинта `final_df_2026_MM`.
Сентябрь нужно сначала собрать прогоном `final_df_script_3_1.ipynb`, иначе ячейка остановится.


In [ ]:
CHOD_COLS = ['report_month', 'inn', 'chod']


def month_key(v):
    s = str(v).strip()
    if len(s) >= 7 and s[4] == '-':
        return s[:7]
    ts = pd.to_datetime(s, errors='coerce')
    return None if pd.isna(ts) else ts.strftime('%Y-%m')


def read_final_df(path):
    if path.suffix.lower() == '.parquet':
        df = pd.read_parquet(path, columns=CHOD_COLS)
    else:
        df = pd.read_csv(path, usecols=CHOD_COLS, dtype={'inn': str}, low_memory=False)
    df['report_month'] = df['report_month'].map(month_key)
    return df


csv_paths = sorted(DATA_DIR.glob(FINAL_DF_CSV_GLOB), key=lambda p: p.stat().st_mtime, reverse=True)
ckpt_dirs = sorted(DATA_DIR.glob(CHECKPOINT_DIR_GLOB), key=lambda p: p.stat().st_mtime, reverse=True)
print('period CSV:', [p.name for p in csv_paths])
print('checkpoint dirs:', [p.name for p in ckpt_dirs])

fd_parts, chod_source = [], {}
if csv_paths:
    fd_csv = read_final_df(csv_paths[0])
    fd_csv = fd_csv.loc[fd_csv['report_month'].isin(MONTHS)]
    fd_parts.append(fd_csv)
    for m in sorted(fd_csv['report_month'].dropna().unique()):
        chod_source[m] = csv_paths[0].name

for m in MONTHS:
    if m in chod_source:
        continue
    key = m.replace('-', '_')
    path = next(
        (d / f'final_df_{key}.{ext}' for d in ckpt_dirs for ext in ('parquet', 'csv')
         if (d / f'final_df_{key}.{ext}').exists()),
        None,
    )
    if path is None:
        continue
    part = read_final_df(path)
    part['report_month'] = part['report_month'].fillna(m)
    fd_parts.append(part.loc[part['report_month'] == m])
    chod_source[m] = f'{path.parent.name}/{path.name}'

missing_chod_months = [m for m in MONTHS if m not in chod_source]
if missing_chod_months:
    raise FileNotFoundError(
        f'Нет final_df за месяцы {missing_chod_months}. Прогоните final_df_script_3_1.ipynb '
        f'на эти месяцы (чекпоинт final_df_2026_MM в {CHECKPOINT_DIR_GLOB}) и перезапустите ячейку.'
    )

fd = pd.concat(fd_parts, ignore_index=True)
fd['inn'] = fd['inn'].map(normalize_inn)
fd['chod'] = pd.to_numeric(fd['chod'], errors='coerce')
chod_df = (
    fd.dropna(subset=['inn'])
    .groupby(['report_month', 'inn'], as_index=False)['chod']
    .sum(min_count=1)
)
chod_map = {(r.report_month, r.inn): r.chod for r in chod_df.itertuples(index=False)}

chod_check = []
for m in MONTHS:
    fd_set = set(chod_df.loc[chod_df['report_month'] == m, 'inn'])
    chod_check.append({
        'Месяц': m,
        'Источник ЧОД': chod_source[m],
        'ИНН в final_df': len(fd_set),
        'ИНН периметра без строки в final_df': len(per_sets[m] - fd_set),
        'ИНН final_df вне периметра': len(fd_set - per_sets[m]),
        'ЧОД ТЭ по периметру, сумма': chod_df.loc[
            (chod_df['report_month'] == m) & chod_df['inn'].isin(per_sets[m]), 'chod'
        ].sum(),
    })
chod_check_df = pd.DataFrame(chod_check)
display(chod_check_df)


## 3) Отчёты коллег

Файлы `??_*_2026.xlsx` в `DATA_DIR`, месяц — по номеру в начале имени. Строка заголовка — первая, где есть ячейка «ИНН».


In [ ]:
excel_paths = {}
for p in sorted(DATA_DIR.glob('??_*_2026.xlsx')):
    mm = re.match(r'^(\d{2})_', p.name)
    if mm and not p.name.startswith('~$'):
        excel_paths.setdefault(f'2026-{mm.group(1)}', p)
excel_paths.update({m: Path(p) for m, p in EXCEL_OVERRIDE.items()})

excel_sets, excel_info = {}, {}
for m in MONTHS:
    p = excel_paths.get(m)
    if p is None or not p.exists():
        excel_info[m] = {'file': None, 'status': 'нет отчёта'}
        print(f'{m}: нет отчёта')
        continue
    try:
        ex, hdr = read_with_header(p)
        inn_col = EXCEL_INN_COL if EXCEL_INN_COL in ex.columns else pick(
            list(ex.columns), ['инн', 'инн клиента'], ('инн',), exclude=('банк',)
        )
        if inn_col is None:
            raise KeyError(f'нет колонки ИНН, колонки: {list(ex.columns)[:30]}')
    except Exception as exc:
        excel_info[m] = {'file': p.name, 'status': f'ошибка чтения: {type(exc).__name__}: {exc}'}
        print(f'{m}: {excel_info[m]["status"]}')
        continue

    raw = ex[inn_col]
    raw = raw[raw.notna() & raw.astype(str).str.strip().ne('')]
    norm = raw.map(normalize_inn)
    excel_sets[m] = set(norm.dropna())
    excel_info[m] = {
        'file': p.name, 'status': 'ok', 'header_row': hdr, 'inn_col': inn_col,
        'rows': len(raw), 'inn': len(excel_sets[m]),
    }
    print(f'{m}: {p.name} | заголовок в строке {hdr} | колонка «{inn_col}» | '
          f'строк с ИНН {len(raw):,} | уникальных ИНН {len(excel_sets[m]):,}')

    bad = pd.DataFrame({'raw': raw, 'inn': norm})
    bad['ok'] = bad['inn'].map(inn_checksum_ok)
    bad = bad.loc[bad['inn'].isna() | ~bad['ok']]
    if len(bad):
        quality_rows.append(pd.DataFrame({
            'Источник': f'отчёт {p.name}',
            'Месяц': m,
            'ИНН сырой': bad['raw'].astype(str),
            'ИНН нормализованный': bad['inn'],
            'Проблема': bad['inn'].map(
                lambda v: 'не 10/12 цифр после нормализации' if pd.isna(v) else 'неверная контрольная сумма'
            ),
            'Наименование': None,
            'Номер договора': None,
        }))


## 4) Сверка по месяцу

Для ИНН «только отчёт» причина по Альфе (первая сработавшая):
нет компании с таким ИНН → компания удалена → нет договоров → нет договора SA → договор SA не действует в месяце →
нет привязки P в месяце. Для ИНН «только периметр» — договоры и признак открытия или закрытия в этом месяце.


In [ ]:
def reasons_sql(inn_variants, month_start, month_end):
    in_list = ', '.join(f"'{v}'" for v in inn_variants)
    return f'''
WITH pt AS (
  SELECT DISTINCT cast(t.n_agr AS string) AS n_agr
  FROM ods_alpha.scd1_agr_terms t
  WHERE cast(t.d_valid_from AS date) <= cast('{month_end}' AS date)
    AND (t.d_valid_to IS NULL OR cast(t.d_valid_to AS date) >= cast('{month_start}' AS date))
    AND upper(trim(cast(t.cf_ter_type AS string))) = 'P'
    AND coalesce(t.ods_deleted_flg, '0') <> '1'
),
x AS (
  SELECT
    regexp_replace(cast(c.c_inn AS string), '[^0-9]', '') AS inn_digits,
    CASE WHEN coalesce(c.ods_deleted_flg, '0') <> '1' THEN 1 ELSE 0 END AS cmp_alive,
    CASE WHEN a.n_agr IS NOT NULL AND coalesce(a.ods_deleted_flg, '0') <> '1' THEN 1 ELSE 0 END AS agr_alive,
    CASE WHEN upper(trim(cast(a.acq_class AS string))) = 'SA' THEN 1 ELSE 0 END AS is_sa,
    CASE WHEN cast(a.d_valid_from AS date) <= cast('{month_end}' AS date)
          AND (a.d_valid_to IS NULL OR cast(a.d_valid_to AS date) >= cast('{month_start}' AS date))
         THEN 1 ELSE 0 END AS in_month,
    CASE WHEN pt.n_agr IS NOT NULL THEN 1 ELSE 0 END AS has_p
  FROM ods_alpha.scd1_companies c
  LEFT JOIN ods_alpha.scd1_agreements a
    ON a.n_cmp_client = c.n_cmp
  LEFT JOIN pt
    ON pt.n_agr = cast(a.n_agr AS string)
  WHERE regexp_replace(cast(c.c_inn AS string), '[^0-9]', '') IN ({in_list})
)
SELECT
  inn_digits,
  max(cmp_alive) AS cmp_alive,
  max(cmp_alive * agr_alive) AS has_agr,
  max(cmp_alive * agr_alive * is_sa) AS has_sa,
  max(cmp_alive * agr_alive * is_sa * in_month) AS has_sa_month,
  max(cmp_alive * agr_alive * is_sa * in_month * has_p) AS has_sa_p_month
FROM x
GROUP BY inn_digits
'''


def reason_text(r):
    if r is None:
        return 'нет компании с таким ИНН в Альфе'
    if not r['cmp_alive']:
        return 'компания удалена'
    if not r['has_agr']:
        return 'у компании нет действующих договоров'
    if not r['has_sa']:
        return 'нет договора SA (есть другого класса)'
    if not r['has_sa_month']:
        return 'договор SA не действует в месяце'
    if not r['has_sa_p_month']:
        return 'договор SA есть, нет привязки P в месяце'
    return 'по Альфе в периметре — проверить ИНН в отчёте'


def only_excel_reasons(inns, month_start, month_end):
    variants = set()
    for i in inns:
        variants.add(i)
        variants.add(i.lstrip('0'))
    parts = []
    for ch in chunks(sorted(variants), IN_CHUNK):
        parts.append(fetch(reasons_sql(ch, month_start, month_end), f'причины {month_start[:7]}'))
    if not parts:
        return {}
    df = pd.concat(parts, ignore_index=True)
    df['inn'] = df['inn_digits'].map(normalize_inn)
    flags = ['cmp_alive', 'has_agr', 'has_sa', 'has_sa_month', 'has_sa_p_month']
    df[flags] = df[flags].apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)
    df = df.dropna(subset=['inn']).groupby('inn')[flags].max()
    return df.to_dict('index')


summary_rows, diff_rows = [], []
for m in MONTHS:
    month_start, month_end = month_bounds(m)
    per_set = per_sets[m]
    info = excel_info.get(m, {})
    row = {
        'Месяц': m,
        'ИНН периметра': len(per_set),
        'Отчёт': info.get('file'),
        'Статус отчёта': info.get('status'),
    }
    if m not in excel_sets:
        summary_rows.append(row)
        continue

    ex_set = excel_sets[m]
    both = per_set & ex_set
    only_per = per_set - ex_set
    only_ex = ex_set - per_set
    row.update({
        'ИНН в отчёте': len(ex_set),
        'В обоих': len(both),
        'Только периметр': len(only_per),
        'Только отчёт': len(only_ex),
        'Совпадение, % от отчёта': round(100.0 * len(both) / len(ex_set), 2) if ex_set else None,
        'Совпадение, % от периметра': round(100.0 * len(both) / len(per_set), 2) if per_set else None,
    })
    summary_rows.append(row)

    if only_per:
        part = per_inn.loc[(per_inn['report_month'] == m) & per_inn['inn'].isin(only_per)].copy()
        part['opened'] = part['d_valid_from'] >= pd.Timestamp(month_start)
        part['closed'] = part['d_valid_to'].notna() & (part['d_valid_to'] <= pd.Timestamp(month_end))
        g = part.groupby('inn').agg(
            company_name=('company_name', 'first'),
            contracts=('contract_number', lambda s: ', '.join(sorted(set(s.dropna().astype(str))))),
            d_from=('d_valid_from', 'min'),
            d_to=('d_valid_to', 'max'),
            opened=('opened', 'any'),
            closed=('closed', 'any'),
        ).reset_index()
        g['reason'] = g.apply(
            lambda r: 'договор открыт в этом месяце' if r['opened']
            else ('договор закрыт в этом месяце' if r['closed'] else 'договор действует весь месяц'),
            axis=1,
        )
        diff_rows.append(pd.DataFrame({
            'Месяц': m,
            'ИНН': g['inn'],
            'Где есть': 'только периметр',
            'Причина': g['reason'],
            'Наименование': g['company_name'],
            'Номер договора': g['contracts'],
            'Дата открытия': g['d_from'].dt.date,
            'Дата закрытия': g['d_to'].dt.date,
        }))

    if only_ex:
        flags = only_excel_reasons(only_ex, month_start, month_end)
        inns = sorted(only_ex)
        diff_rows.append(pd.DataFrame({
            'Месяц': m,
            'ИНН': inns,
            'Где есть': 'только отчёт',
            'Причина': [reason_text(flags.get(i)) for i in inns],
            'Наименование': None,
            'Номер договора': None,
            'Дата открытия': None,
            'Дата закрытия': None,
        }))
    print(f'{m}: в обоих {len(both):,} | только периметр {len(only_per):,} | только отчёт {len(only_ex):,}')

summary = pd.DataFrame(summary_rows)
diff_df = pd.concat(diff_rows, ignore_index=True) if diff_rows else pd.DataFrame(
    columns=['Месяц', 'ИНН', 'Где есть', 'Причина', 'Наименование', 'Номер договора',
             'Дата открытия', 'Дата закрытия']
)
display(summary)
if len(diff_df):
    print('Причины расхождений:')
    display(diff_df.groupby(['Месяц', 'Где есть', 'Причина']).size().rename('ИНН').reset_index())


## 5) Excel: листы месяцев (ИНН, ЧОД ТЭ), Итого, Расхождения, Качество ИНН

In [ ]:
quality_df = pd.concat(quality_rows, ignore_index=True) if quality_rows else pd.DataFrame(
    columns=['Источник', 'Месяц', 'ИНН сырой', 'ИНН нормализованный', 'Проблема',
             'Наименование', 'Номер договора']
)

empty = [m for m in MONTHS if not per_sets[m]]
if empty:
    raise RuntimeError(f'Пустой периметр за месяцы: {empty}')


def text_cols(ws, df, cols, width=16):
    for c in cols:
        if c not in df.columns:
            continue
        idx = df.columns.get_loc(c) + 1
        letter = ws.cell(row=1, column=idx).column_letter
        ws.column_dimensions[letter].width = width
        for (cell,) in ws.iter_rows(min_row=2, min_col=idx, max_col=idx):
            cell.number_format = '@'


summary_out = summary.merge(chod_check_df, on='Месяц', how='left')
diff_out = diff_df.copy()
diff_out['ЧОД ТЭ'] = [chod_map.get((m, i)) for m, i in zip(diff_out['Месяц'], diff_out['ИНН'])]

with pd.ExcelWriter(XLSX_PATH, engine='openpyxl') as writer:
    summary_out.to_excel(writer, sheet_name='Итого', index=False)
    for m in MONTHS:
        inns = sorted(per_sets[m])
        part = pd.DataFrame({'ИНН': inns, 'ЧОД ТЭ': [chod_map.get((m, i)) for i in inns]})
        part.to_excel(writer, sheet_name=m, index=False)
        writer.sheets[m].column_dimensions['B'].width = 16
        for (cell,) in writer.sheets[m].iter_rows(min_row=2, min_col=2, max_col=2):
            cell.number_format = '#,##0.00'
        text_cols(writer.sheets[m], part, ['ИНН'])
    diff_out.to_excel(writer, sheet_name='Расхождения', index=False)
    text_cols(writer.sheets['Расхождения'], diff_out, ['ИНН'])
    quality_df.to_excel(writer, sheet_name='Качество ИНН', index=False)
    text_cols(writer.sheets['Качество ИНН'], quality_df, ['ИНН сырой', 'ИНН нормализованный'])

print('Saved:', XLSX_PATH)


## 6) Файл для коллеги: строка на ИНН, ЧОД ТЭ по месяцам в колонках

ИНН — все, кто хотя бы в одном месяце был в периметре. 0 — клиента не было в периметре в этом месяце
или у него нет строки в `final_df`.


In [ ]:
all_inns = sorted(set().union(*per_sets.values()))
wide = pd.DataFrame({'ИНН': all_inns})
month_cols = []
for m in MONTHS:
    col = f'ЧОД ТЭ {m[5:7]}.{m[:4]}'
    wide[col] = [chod_map.get((m, i)) for i in all_inns]
    month_cols.append(col)
wide[month_cols] = wide[month_cols].apply(pd.to_numeric, errors='coerce').fillna(0).round(2)

with pd.ExcelWriter(CLIENT_XLSX_PATH, engine='openpyxl') as writer:
    wide.to_excel(writer, sheet_name='ИНН и ЧОД ТЭ', index=False)
    ws = writer.sheets['ИНН и ЧОД ТЭ']
    text_cols(ws, wide, ['ИНН'])
    for col in month_cols:
        idx = wide.columns.get_loc(col) + 1
        ws.column_dimensions[ws.cell(row=1, column=idx).column_letter].width = 16
        for (cell,) in ws.iter_rows(min_row=2, min_col=idx, max_col=idx):
            cell.number_format = '#,##0.00'
    ws.freeze_panes = 'B2'

print('ИНН:', f'{len(wide):,}')
display(wide[month_cols].sum().rename('ЧОД ТЭ, сумма').to_frame())
print('Saved:', CLIENT_XLSX_PATH)
